# Instructor Demo: Polars Foundations

**Companion demo for the Pandas Deep Dive.** This notebook teaches the Polars foundations students need after seeing pandas: expressions, eager vs lazy execution, strict typing, fast grouped analytics, time windows, and pipeline-style data engineering code.

Teaching arc: **pandas mental model -> Polars expression model -> SQL / warehouse bridge**. Use it live: ask the room what each expression means before running the context that executes it.

| Level | Sections |
|---|---|
| Beginner | 1 to 7: structures, expressions, reading, inspecting, selecting, filtering, sorting |
| Intermediate | 8 to 15: cleaning, types, strings, new columns, immutability, groupby, windows, joins |
| Advanced | 16 to 21: reshaping, time series, lazy execution, query plans, performance, interop |

Kernel: a Python environment with `polars` installed. The notebook is designed for **Polars 1.x** and uses a small messy taxi CSV like the pandas demo. If the class repo data file is not found, the notebook writes a fallback 16-row CSV so the demo remains runnable.

In [1]:
# In a fresh environment, install first:
# pip install polars pyarrow pandas

from __future__ import annotations

from pathlib import Path
from time import perf_counter
from datetime import datetime, timedelta, timezone
import random
import textwrap

import polars as pl
import polars.selectors as cs

print("polars:", pl.__version__)
print("thread pool size:", pl.thread_pool_size())

pl.Config.set_tbl_rows(8)
pl.Config.set_tbl_cols(14)
pl.Config.set_fmt_str_lengths(42)

polars: 1.42.1
thread pool size: 10


polars.config.Config

## Why Polars after pandas? Talk track

Polars is not "pandas with different method names." The important shift is:

1. **Expression-first:** `pl.col("fare") + pl.col("tip")` describes work. A context such as `select`, `with_columns`, `filter`, or `group_by` runs it.
2. **No hidden index:** rows are rows; if row identity matters, create a row number explicitly.
3. **Strict types by default:** bad parses should become visible. Use `strict=False` when the cleaning policy is "invalid values become null."
4. **Lazy execution exists from day one:** `scan_csv` builds a query plan; `.collect()` executes it.
5. **Columnar/Arrow-native thinking:** Polars is built for columnar analytics, projection pushdown, predicate pushdown, parallelism, and streaming-friendly pipelines.

Instructor bridge: pandas 3 introduced `pd.col`. Polars students should recognize the same idea immediately: **expressions are the unit of transformation**.

---
# BEGINNER

## 1. Series and DataFrame

Polars has Series and DataFrames like pandas, but the daily teaching unit is usually the **expression**, not the individual Series. Start with the familiar structures, then move quickly to `pl.col`.

In [2]:
fares = pl.Series("fare_amount", [18.5, 22.0, 16.25])
fares

fare_amount
f64
18.5
22.0
16.25


In [3]:
trips = pl.DataFrame(
    {
        "trip_id": ["T001", "T002", "T003"],
        "pickup_borough": ["Manhattan", "Queens", "Bronx"],
        "fare_amount": [18.5, 35.0, 22.0],
        "tip_amount": [3.0, 5.25, 0.0],
    }
)

trips

trip_id,pickup_borough,fare_amount,tip_amount
str,str,f64,f64
"""T001""","""Manhattan""",18.5,3.0
"""T002""","""Queens""",35.0,5.25
"""T003""","""Bronx""",22.0,0.0


In [4]:
trips.schema

Schema([('trip_id', String),
        ('pickup_borough', String),
        ('fare_amount', Float64),
        ('tip_amount', Float64)])

Pause point: Polars shows dtypes directly in the table output. Use this early. In Polars, schema awareness is not a side topic; it is part of how you write correct pipelines.

## 2. Expressions and Contexts

A Polars expression is a **recipe**, not a computed result. This is the first key mental model.

`pl.col("fare_amount") + pl.col("tip_amount")` does not read data by itself. It must be placed in a context:

- `select`: produce selected/new columns only
- `with_columns`: keep existing columns and add/replace columns
- `filter`: keep rows where a Boolean expression is true
- `group_by().agg`: aggregate per group

In [5]:
total_expr = (pl.col("fare_amount") + pl.col("tip_amount")).alias("total_amount")
total_expr

<Expr ['[(col("fare_amount")) + (col("…'] at 0x105AB0550>

In [6]:
# select: output only what you ask for.
trips.select(
    "trip_id",
    total_expr,
)

trip_id,total_amount
str,f64
"""T001""",21.5
"""T002""",40.25
"""T003""",22.0


In [7]:
# with_columns: keep the original table and add/replace columns.
trips.with_columns(
    total_expr,
    tip_rate=(pl.col("tip_amount") / pl.col("fare_amount")).round(3),
)

trip_id,pickup_borough,fare_amount,tip_amount,total_amount,tip_rate
str,str,f64,f64,f64,f64
"""T001""","""Manhattan""",18.5,3.0,21.5,0.162
"""T002""","""Queens""",35.0,5.25,40.25,0.15
"""T003""","""Bronx""",22.0,0.0,22.0,0.0


In [8]:
# filter: expression returns True/False for each row.
trips.filter(pl.col("fare_amount") > 20)

trip_id,pickup_borough,fare_amount,tip_amount
str,str,f64,f64
"""T002""","""Queens""",35.0,5.25
"""T003""","""Bronx""",22.0,0.0


Instructor framing: Polars code often reads like SQL because you are describing a query. `select`, `filter`, `group_by`, and `join` should sound familiar before students reach BigQuery.

## 3. Reading Data

We use the same messy taxi-review shape as the pandas demo: invalid fare, invalid timestamp, missing tip, missing borough, and one negative distance. The point is to show that real data forces type decisions.

For messy CSV demos, this notebook first reads everything as strings with `infer_schema=False`. That makes the raw import reliable, then Section 8 shows explicit coercion.

In [9]:
candidates = [
    Path("Week 3/Labs/Day 1/data/taxi_trip_review.csv"),  # opened from repo root
    Path("../Labs/Day 1/data/taxi_trip_review.csv"),       # opened from Instructor Notes
    Path("data/taxi_trip_review.csv"),                     # copied next to this notebook
]

existing = [p for p in candidates if p.exists()]

if existing:
    DATA_PATH = existing[0]
else:
    DATA_PATH = Path("data/taxi_trip_review.csv")
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    DATA_PATH.write_text('trip_id,vendor,pickup_ts,pickup_borough,dropoff_borough,trip_distance,fare_amount,tip_amount,payment_type\nT001,CMT,2026-06-01 08:15:00,Manhattan,Brooklyn,3.2,18.50,3.00,card\nT002,VTS,2026-06-01 09:05:00,Queens,Manhattan,8.1,35.00,5.25,card\nT003,CMT,2026-06-01 10:20:00,Bronx,Manhattan,5.5,22.00,0.00,cash\nT004,VTS,not-a-date,Brooklyn,Queens,2.4,abc,1.00,card\nT005,CMT,2026-06-01 12:45:00,,Manhattan,1.8,9.50,,cash\nT006,VTS,2026-06-02 07:30:00,Manhattan,Queens,-1.0,14.00,2.00,card\nT007,CMT,2026-06-02 08:10:00,Manhattan,Bronx,4.9,21.75,4.00,card\nT008,VTS,2026-06-02 09:25:00,Staten Island,Brooklyn,7.3,28.40,0.00,cash\nT009,CMT,2026-06-02 11:40:00,Queens,Queens,1.2,8.75,1.25,card\nT010,VTS,2026-06-02 13:00:00,Brooklyn,Manhattan,6.8,31.20,3.10,voucher\nT011,CMT,2026-06-03 14:15:00,Bronx,Queens,4.2,19.00,2.50,card\nT012,VTS,2026-06-03 15:50:00,Manhattan,Manhattan,0.9,7.50,1.00,cash\nT013,CMT,2026-06-03 16:30:00,Queens,Airport,10.5,42.00,8.00,card\nT014,VTS,2026-06-03 18:05:00,Brooklyn,Brooklyn,2.7,13.25,2.00,card\nT015,CMT,2026-06-03 19:10:00,Manhattan,Airport,12.0,48.00,9.50,card\nT016,VTS,2026-06-03 20:40:00,Bronx,Bronx,1.5,11.00,0.00,cash\n', encoding="utf-8")

print("using:", DATA_PATH)

df_raw = pl.read_csv(
    DATA_PATH,
    infer_schema=False,     # safe first read for messy files
    null_values=[""],       # empty CSV fields become null
)

df_raw.head()

using: ../Labs/Day 1/data/taxi_trip_review.csv


trip_id,vendor,pickup_borough,dropoff_borough,fare_amount,tip_amount,trip_distance,pickup_ts,payment_type
str,str,str,str,str,str,str,str,str
"""T001""","""VTS""","""Manhattan""","""Queens""","""18.50""","""3.70""","""5.2""","""2026-07-06T08:15:00Z""","""card"""
"""T002""","""CMT""","""Manhattan""","""Brooklyn""","""22.00""","""0.00""","""7.1""","""2026-07-06T08:40:00Z""","""cash"""
"""T003""","""VTS""","""Queens""","""Manhattan""","""16.25""","""2.50""","""4.4""","""2026-07-06T09:05:00Z""","""card"""
"""T004""","""CMT""","""Bronx""","""Manhattan""","""28.00""","""4.20""","""9.0""","""2026-07-06T09:20:00Z""","""card"""
"""T005""","""VTS""","""Brooklyn""","""Queens""","""14.00""","""1.50""","""3.2""","""2026-07-06T10:00:00Z""","""card"""


In [10]:
df_raw.schema

Schema([('trip_id', String),
        ('vendor', String),
        ('pickup_borough', String),
        ('dropoff_borough', String),
        ('fare_amount', String),
        ('tip_amount', String),
        ('trip_distance', String),
        ('pickup_ts', String),
        ('payment_type', String)])

## 4. Inspecting: the First Five Minutes with Any Dataset

Profile before touching anything. In Polars, use `shape`, `schema`, `glimpse`, `describe`, `null_count`, and quick frequency counts.

In [11]:
df_raw.shape

(16, 9)

In [12]:
df_raw.glimpse()

Rows: 16
Columns: 9
$ trip_id         <str> 'T001', 'T002', 'T003', 'T004', 'T005', 'T006', 'T007', 'T008', 'T009', 'T010'
$ vendor          <str> 'VTS', 'CMT', 'VTS', 'CMT', 'VTS', 'CMT', 'VTS', 'CMT', 'VTS', 'CMT'
$ pickup_borough  <str> 'Manhattan', 'Manhattan', 'Queens', 'Bronx', 'Brooklyn', 'Queens', 'Manhattan', 'Staten Island', 'Brooklyn', 'Queens'
$ dropoff_borough <str> 'Queens', 'Brooklyn', 'Manhattan', 'Manhattan', 'Queens', 'Brooklyn', 'Manhattan', 'Manhattan', 'Manhattan', 'Queens'
$ fare_amount     <str> '18.50', '22.00', '16.25', '28.00', '14.00', '19.75', '9.50', '35.00', 'abc', '11.00'
$ tip_amount      <str> '3.70', '0.00', '2.50', '4.20', '1.50', null, '1.00', '5.00', '2.00', '-1.00'
$ trip_distance   <str> '5.2', '7.1', '4.4', '9.0', '3.2', '6.3', '1.4', '12.2', '5.0', '2.5'
$ pickup_ts       <str> '2026-07-06T08:15:00Z', '2026-07-06T08:40:00Z', '2026-07-06T09:05:00Z', '2026-07-06T09:20:00Z', '2026-07-06T10:00:00Z', '2026-07-06T10:30:00Z', 'not-a-date', '2026-07-06T

In [13]:
df_raw.describe()

statistic,trip_id,vendor,pickup_borough,dropoff_borough,fare_amount,tip_amount,trip_distance,pickup_ts,payment_type
str,str,str,str,str,str,str,str,str,str
"""count""","""16""","""16""","""15""","""16""","""16""","""15""","""16""","""16""","""16"""
"""null_count""","""0""","""0""","""1""","""0""","""0""","""1""","""0""","""0""","""0"""
"""mean""",null,null,null,null,null,null,null,null,null
"""std""",null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…
"""25%""",null,null,null,null,null,null,null,null,null
"""50%""",null,null,null,null,null,null,null,null,null
"""75%""",null,null,null,null,null,null,null,null,null
"""max""","""T015""","""VTS""","""Staten Island""","""Queens""","""abc""","""5.00""","""9.0""","""not-a-date""","""cash"""


In [14]:
df_raw.null_count()

trip_id,vendor,pickup_borough,dropoff_borough,fare_amount,tip_amount,trip_distance,pickup_ts,payment_type
u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,1,0,0,1,0,0,0


In [15]:
df_raw.get_column("pickup_borough").value_counts(sort=True)

pickup_borough,count
str,u32
"""Manhattan""",6
"""Queens""",3
"""Brooklyn""",3
"""Bronx""",2
"""Staten Island""",1
null,1


Pause point: because we read as strings, `describe()` is less numerically useful at first. That is intentional: the raw landing layer is safe, but not analytics-ready. Cleaning turns strings into usable types.

## 5. Selecting Data

Polars does not have `.loc` and `.iloc` as the main classroom vocabulary. Teach:

- `select(...)` for columns and expressions
- `slice(offset, length)` for row positions
- `with_row_index(...)` when row identity must be explicit

In [16]:
df_raw.select("trip_id", "fare_amount").head(3)

trip_id,fare_amount
str,str
"""T001""","""18.50"""
"""T002""","""22.00"""
"""T003""","""16.25"""


In [17]:
df_raw.select(
    pl.col("trip_id"),
    pl.col("pickup_borough"),
    pl.col("payment_type"),
).head(3)

trip_id,pickup_borough,payment_type
str,str,str
"""T001""","""Manhattan""","""card"""
"""T002""","""Manhattan""","""cash"""
"""T003""","""Queens""","""card"""


In [18]:
df_raw.slice(0, 3).select("trip_id", "pickup_borough", "fare_amount")

trip_id,pickup_borough,fare_amount
str,str,str
"""T001""","""Manhattan""","""18.50"""
"""T002""","""Manhattan""","""22.00"""
"""T003""","""Queens""","""16.25"""


In [19]:
df_raw.with_row_index("row_nr").head(5)

row_nr,trip_id,vendor,pickup_borough,dropoff_borough,fare_amount,tip_amount,trip_distance,pickup_ts,payment_type
u32,str,str,str,str,str,str,str,str,str
0,"""T001""","""VTS""","""Manhattan""","""Queens""","""18.50""","""3.70""","""5.2""","""2026-07-06T08:15:00Z""","""card"""
1,"""T002""","""CMT""","""Manhattan""","""Brooklyn""","""22.00""","""0.00""","""7.1""","""2026-07-06T08:40:00Z""","""cash"""
2,"""T003""","""VTS""","""Queens""","""Manhattan""","""16.25""","""2.50""","""4.4""","""2026-07-06T09:05:00Z""","""card"""
3,"""T004""","""CMT""","""Bronx""","""Manhattan""","""28.00""","""4.20""","""9.0""","""2026-07-06T09:20:00Z""","""card"""
4,"""T005""","""VTS""","""Brooklyn""","""Queens""","""14.00""","""1.50""","""3.2""","""2026-07-06T10:00:00Z""","""card"""


Instructor bridge: "No hidden index" is a gift for data engineering. In production pipelines, row identity should be a real column, not an implicit notebook convenience.

## 6. Filtering

Polars filters with expressions. Combine conditions with `&` and `|`, using parentheses just like pandas boolean masks.

In [20]:
is_manhattan = pl.col("pickup_borough") == "Manhattan"

df_raw.select(
    "trip_id",
    "pickup_borough",
    is_manhattan.alias("is_manhattan"),
).head(8)

trip_id,pickup_borough,is_manhattan
str,str,bool
"""T001""","""Manhattan""",true
"""T002""","""Manhattan""",true
"""T003""","""Queens""",false
"""T004""","""Bronx""",false
"""T005""","""Brooklyn""",false
"""T006""","""Queens""",false
"""T007""","""Manhattan""",true
"""T008""","""Staten Island""",false


In [21]:
df_raw.filter(
    (pl.col("pickup_borough") == "Manhattan")
    & (pl.col("payment_type") == "card")
)

trip_id,vendor,pickup_borough,dropoff_borough,fare_amount,tip_amount,trip_distance,pickup_ts,payment_type
str,str,str,str,str,str,str,str,str
"""T001""","""VTS""","""Manhattan""","""Queens""","""18.50""","""3.70""","""5.2""","""2026-07-06T08:15:00Z""","""card"""
"""T007""","""VTS""","""Manhattan""","""Manhattan""","""9.50""","""1.00""","""1.4""","""not-a-date""","""card"""
"""T011""","""VTS""","""Manhattan""","""Bronx""","""24.50""","""3.00""","""-4.0""","""2026-07-06T12:30:00Z""","""card"""
"""T001""","""VTS""","""Manhattan""","""Queens""","""18.50""","""3.70""","""5.2""","""2026-07-06T08:15:00Z""","""card"""
"""T014""","""CMT""","""Manhattan""","""Queens""","""17.00""","""2.00""","""4.9""","""2026-07-06T14:00:00Z""","""card"""


In [22]:
df_raw.filter(pl.col("pickup_borough").is_in(["Bronx", "Staten Island"]))

trip_id,vendor,pickup_borough,dropoff_borough,fare_amount,tip_amount,trip_distance,pickup_ts,payment_type
str,str,str,str,str,str,str,str,str
"""T004""","""CMT""","""Bronx""","""Manhattan""","""28.00""","""4.20""","""9.0""","""2026-07-06T09:20:00Z""","""card"""
"""T008""","""CMT""","""Staten Island""","""Manhattan""","""35.00""","""5.00""","""12.2""","""2026-07-06T11:15:00Z""","""card"""
"""T013""","""VTS""","""Bronx""","""Queens""","""21.25""","""2.25""","""6.1""","""2026-07-06T13:15:00Z""","""card"""


In [23]:
# This works because we cast inside the filter expression.
df_raw.filter(
    pl.col("trip_distance").cast(pl.Float64, strict=False).is_between(4, 8)
).select("trip_id", "pickup_borough", "trip_distance", "fare_amount")

trip_id,pickup_borough,trip_distance,fare_amount
str,str,str,str
"""T001""","""Manhattan""","""5.2""","""18.50"""
"""T002""","""Manhattan""","""7.1""","""22.00"""
"""T003""","""Queens""","""4.4""","""16.25"""
"""T006""","""Queens""","""6.3""","""19.75"""
…,…,…,…
"""T013""","""Bronx""","""6.1""","""21.25"""
"""T001""","""Manhattan""","""5.2""","""18.50"""
"""T014""","""Manhattan""","""4.9""","""17.00"""
"""T015""",null,"""4.9""","""17.00"""


In [24]:
# Strictness demo: raw fare_amount is text, so numeric comparison is not valid yet.
try:
    df_raw.filter(pl.col("fare_amount") > 15)
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc).splitlines()[0])

ComputeError
cannot compare string with numeric type (i32)


Pause point: this is a feature, not an annoyance. Polars makes the type problem visible instead of silently giving a wrong numeric answer on text.

## 7. Sorting and Top-N

A deliberate trap: sorting a text column sorts lexicographically, not numerically. Show the wrong result first, then fix it with a cast.

In [25]:
# Wrong for numeric analysis: fare_amount is still text.
df_raw.sort("fare_amount", descending=True).select("trip_id", "fare_amount").head(5)

trip_id,fare_amount
str,str
"""T009""","""abc"""
"""T007""","""9.50"""
"""T012""","""8.75"""
"""T008""","""35.00"""
"""T004""","""28.00"""


In [26]:
# Right for numeric analysis: cast first.
df_raw.with_columns(
    fare_amount_num=pl.col("fare_amount").cast(pl.Float64, strict=False)
).sort("fare_amount_num", descending=True).select(
    "trip_id", "fare_amount", "fare_amount_num"
).head(5)

trip_id,fare_amount,fare_amount_num
str,str,f64
"""T009""","""abc""",null
"""T008""","""35.00""",35.0
"""T004""","""28.00""",28.0
"""T011""","""24.50""",24.5
"""T002""","""22.00""",22.0


---
# INTERMEDIATE

## 8. Missing Data and Type Coercion

Polars uses strict typing. The most important cleaning pattern:

1. Read raw safely.
2. Parse/cast deliberately.
3. Use `strict=False` when bad values should become null.
4. Count nulls after coercion.
5. Apply a clear business rule.

In [27]:
clean = df_raw.with_columns(
    fare_amount=pl.col("fare_amount").cast(pl.Float64, strict=False),
    tip_amount=pl.col("tip_amount").cast(pl.Float64, strict=False),
    trip_distance=pl.col("trip_distance").cast(pl.Float64, strict=False),
    pickup_ts=pl.col("pickup_ts").str.to_datetime(
        "%Y-%m-%d %H:%M:%S",
        time_zone="UTC",
        strict=False,
    ),
)

clean.select(
    pl.all().null_count()
)

trip_id,vendor,pickup_borough,dropoff_borough,fare_amount,tip_amount,trip_distance,pickup_ts,payment_type
u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,1,0,1,1,0,16,0


In [28]:
# Policy examples. Say the policy out loud.
clean = (
    clean
    .with_columns(
        tip_amount=pl.col("tip_amount").fill_null(0.0)   # missing tip means no tip
    )
    .drop_nulls(subset=["fare_amount", "pickup_ts", "pickup_borough"])  # required fields
)

print("rows after required-field policy:", clean.height)
clean.head()

rows after required-field policy: 0


trip_id,vendor,pickup_borough,dropoff_borough,fare_amount,tip_amount,trip_distance,pickup_ts,payment_type
str,str,str,str,f64,f64,f64,"datetime[μs, UTC]",str


In [29]:
# Optional data-quality policy: keep the suspicious negative distance visible first.
clean.filter(pl.col("trip_distance") < 0)

trip_id,vendor,pickup_borough,dropoff_borough,fare_amount,tip_amount,trip_distance,pickup_ts,payment_type
str,str,str,str,f64,f64,f64,"datetime[μs, UTC]",str


## 9. dtypes, Categorical, and Schema Thinking

In pandas, students often discover dtypes late. In Polars, dtypes are central. Low-cardinality strings such as payment type and borough can be cast to `Categorical` when useful for repeated analytics.

In [30]:
clean.schema

Schema([('trip_id', String),
        ('vendor', String),
        ('pickup_borough', String),
        ('dropoff_borough', String),
        ('fare_amount', Float64),
        ('tip_amount', Float64),
        ('trip_distance', Float64),
        ('pickup_ts', Datetime(time_unit='us', time_zone='UTC')),
        ('payment_type', String)])

In [31]:
cat_demo = clean.with_columns(
    payment_type_cat=pl.col("payment_type").cast(pl.Categorical)
)

cat_demo.select("payment_type", "payment_type_cat").head()

payment_type,payment_type_cat
str,cat


In [32]:
cat_demo.get_column("payment_type_cat").cat.get_categories()

payment_type_cat
str


Instructor note: keep `Categorical` as a performance/schema concept, not a requirement for every small demo. The bigger point is that production data engineering code should say what types it expects.

## 10. String Methods

String methods live under `.str`. They are expression methods, so they work inside `select`, `with_columns`, `filter`, and lazy pipelines.

In [33]:
demo = pl.DataFrame({"borough": ["  Manhattan ", "queens", "BRONX  "]})

demo.select(
    original=pl.col("borough"),
    cleaned=pl.col("borough").str.strip_chars().str.to_uppercase(),
)

original,cleaned
str,str
""" Manhattan ""","""MANHATTAN"""
"""queens""","""QUEENS"""
"""BRONX ""","""BRONX"""


In [34]:
clean = clean.with_columns(
    vendor=pl.col("vendor").str.strip_chars().str.to_uppercase(),
    payment_type=pl.col("payment_type").str.strip_chars().str.to_lowercase(),
)

clean.select("trip_id", "vendor", "payment_type").head()

trip_id,vendor,payment_type
str,str,str


In [35]:
clean.filter(
    pl.col("dropoff_borough").str.contains("Airport")
).select("trip_id", "pickup_borough", "dropoff_borough", "fare_amount")

trip_id,pickup_borough,dropoff_borough,fare_amount
str,str,str,f64


## 11. New Columns and Conditional Logic

This is where Polars should feel natural after `pd.col`: every new column is an expression.

In [36]:
clean = clean.with_columns(
    total_amount=(pl.col("fare_amount") + pl.col("tip_amount")).round(2),
    tip_rate=(pl.col("tip_amount") / pl.col("fare_amount")).round(3),
    long_trip=pl.col("trip_distance") >= 5,
)

clean.select("trip_id", "fare_amount", "tip_amount", "total_amount", "tip_rate", "long_trip").head()

trip_id,fare_amount,tip_amount,total_amount,tip_rate,long_trip
str,f64,f64,f64,f64,bool


In [37]:
clean.with_columns(
    payment_label=(
        pl.when(pl.col("payment_type") == "card")
        .then(pl.lit("Card payment"))
        .when(pl.col("payment_type") == "cash")
        .then(pl.lit("Cash payment"))
        .otherwise(pl.lit("Other payment"))
    )
).select("trip_id", "payment_type", "payment_label").head(8)

trip_id,payment_type,payment_label
str,str,str


Avoid Python row functions for ordinary arithmetic and rules. `with_columns` expressions are the production habit; they can be optimized and parallelized.

## 12. Immutability: Reassign or Pipeline

Polars methods usually return a new DataFrame. Do not teach "modify a view" or chained assignment. Teach this rule:

**Build a new table, assign it to a name when you want to keep it.**

In [38]:
changed = clean.with_columns(
    fare_amount=pl.lit(0.0)
)

print("changed max fare:", changed.get_column("fare_amount").max())
print("original max fare still intact:", clean.get_column("fare_amount").max())

changed max fare: None
original max fare still intact: None


In [39]:
# The production pattern: reassign intentionally.
capped = clean.with_columns(
    fare_amount=pl.when(pl.col("fare_amount") > 30)
    .then(30.0)
    .otherwise(pl.col("fare_amount"))
)

capped.select("trip_id", "fare_amount").sort("fare_amount", descending=True).head(5)

trip_id,fare_amount
str,f64


Talk track: pandas had to solve view/copy ambiguity. Polars largely avoids that teaching burden by making transformation pipelines explicit.

## 13. GroupBy: Split, Apply, Combine

Same conceptual pattern as pandas and SQL, but the aggregation is expression-based. Use named aliases so the output is production-readable.

In [40]:
summary = (
    clean
    .group_by("pickup_borough")
    .agg(
        trip_count=pl.len(),
        total_revenue=pl.col("total_amount").sum(),
        average_fare=pl.col("fare_amount").mean(),
        average_tip_rate=pl.col("tip_rate").mean(),
    )
    .with_columns(
        pl.col("total_revenue", "average_fare", "average_tip_rate").round(2)
    )
    .sort("total_revenue", descending=True)
)

summary

pickup_borough,trip_count,total_revenue,average_fare,average_tip_rate
str,u32,f64,f64,f64


In [41]:
# Group by an expression, not just a raw column.
clean.group_by(
    pl.col("pickup_ts").dt.date().alias("pickup_date")
).agg(
    trips=pl.len(),
    revenue=pl.col("total_amount").sum().round(2),
).sort("pickup_date")

pickup_date,trips,revenue
date,u32,f64


## 14. Window Expressions with `over`

In pandas this idea often appears as `groupby().transform(...)`. In SQL it becomes `AVG(fare) OVER (PARTITION BY borough)`. In Polars, use `.over(...)`.

In [42]:
borough_avg = pl.col("fare_amount").mean().over("pickup_borough")

clean.with_columns(
    borough_avg_fare=borough_avg.round(2),
    vs_borough_avg=(pl.col("fare_amount") - borough_avg).round(2),
).select(
    "trip_id", "pickup_borough", "fare_amount", "borough_avg_fare", "vs_borough_avg"
).sort("pickup_borough", "trip_id")

trip_id,pickup_borough,fare_amount,borough_avg_fare,vs_borough_avg
str,str,f64,f64,f64


In [43]:
# Windowed rank: top fares within each pickup borough.
clean.with_columns(
    borough_fare_rank=pl.col("fare_amount").rank(descending=True).over("pickup_borough")
).select(
    "pickup_borough", "trip_id", "fare_amount", "borough_fare_rank"
).sort("pickup_borough", "borough_fare_rank")

pickup_borough,trip_id,fare_amount,borough_fare_rank
str,str,f64,f64


## 15. Combining Tables: joins and concat

`join` is SQL JOIN. `concat` stacks compatible tables. This is the same warehouse vocabulary students will use in BigQuery.

In [44]:
zones = pl.DataFrame(
    {
        "borough": ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"],
        "zone_group": ["Core", "Core", "Airport-heavy", "North", "South"],
    }
)

joined = clean.join(
    zones,
    left_on="pickup_borough",
    right_on="borough",
    how="left",
)

joined.select("trip_id", "pickup_borough", "zone_group", "fare_amount").head(8)

trip_id,pickup_borough,zone_group,fare_amount
str,str,str,f64


In [45]:
partial_zones = zones.filter(pl.col("borough") != "Queens")

left = clean.join(partial_zones, left_on="pickup_borough", right_on="borough", how="left")
inner = clean.join(partial_zones, left_on="pickup_borough", right_on="borough", how="inner")

print("left keeps all trips:", left.height)
print("inner drops Queens trips:", inner.height)

left keeps all trips: 0
inner drops Queens trips: 0


In [46]:
jan = clean.head(3)
feb = clean.tail(3)

pl.concat([jan, feb], how="vertical").select("trip_id", "pickup_borough", "fare_amount")

trip_id,pickup_borough,fare_amount
str,str,f64


---
# ADVANCED

## 16. Reshaping: pivot and unpivot

Wide vs long. Analysts often ask for wide tables; data pipelines, BI models, and ML workflows often prefer long/tidy tables.

In [47]:
pivot = (
    clean
    .pivot(
        on="payment_type",
        index="pickup_borough",
        values="fare_amount",
        aggregate_function="mean",
    )
    .with_columns(cs.numeric().round(2))
)

pivot

pickup_borough
str


In [48]:
long_again = (
    pivot
    .unpivot(
        on=cs.numeric(),
        index="pickup_borough",
        variable_name="payment_type",
        value_name="avg_fare",
    )
    .drop_nulls(subset=["avg_fare"])
    .sort("pickup_borough", "payment_type")
)

long_again

pickup_borough,payment_type,avg_fare
str,str,null


## 17. Time Series: dynamic grouping and rolling windows

Polars uses `group_by_dynamic` for time buckets. The index column must be sorted for dynamic grouping.

In [49]:
rng = random.Random(7)

hourly = pl.DataFrame(
    {
        "pickup_ts": [
            datetime(2026, 6, 1, tzinfo=timezone.utc) + timedelta(hours=i)
            for i in range(24 * 14)
        ],
        "rides": [rng.randint(25, 60) for _ in range(24 * 14)],
    }
)

hourly.head(3)

pickup_ts,rides
"datetime[μs, UTC]",i64
2026-06-01 00:00:00 UTC,45
2026-06-01 01:00:00 UTC,34
2026-06-01 02:00:00 UTC,50


In [50]:
daily = (
    hourly
    .sort("pickup_ts")
    .group_by_dynamic("pickup_ts", every="1d")
    .agg(rides=pl.col("rides").sum())
    .sort("pickup_ts")
)

daily.head(7)

pickup_ts,rides
"datetime[μs, UTC]",i64
2026-06-01 00:00:00 UTC,945
2026-06-02 00:00:00 UTC,1013
2026-06-03 00:00:00 UTC,1083
2026-06-04 00:00:00 UTC,1026
2026-06-05 00:00:00 UTC,1069
2026-06-06 00:00:00 UTC,1007
2026-06-07 00:00:00 UTC,932


In [51]:
daily.with_columns(
    rides_7d_avg=pl.col("rides").rolling_mean(window_size=7).round(1)
).tail(8)

pickup_ts,rides,rides_7d_avg
"datetime[μs, UTC]",i64,f64
2026-06-07 00:00:00 UTC,932,1010.7
2026-06-08 00:00:00 UTC,1053,1026.1
2026-06-09 00:00:00 UTC,1096,1038.0
2026-06-10 00:00:00 UTC,996,1025.6
2026-06-11 00:00:00 UTC,1028,1025.9
2026-06-12 00:00:00 UTC,994,1015.1
2026-06-13 00:00:00 UTC,1075,1024.9
2026-06-14 00:00:00 UTC,1102,1049.1


## 18. Ordering Operations: shift, diff, rank, cumsum

These are the bridge to SQL window functions: `LAG`, day-over-day difference, running totals, and ranks.

In [52]:
daily.with_columns(
    prev_day=pl.col("rides").shift(1),
    day_change=pl.col("rides").diff(),
    running_total=pl.col("rides").cum_sum(),
    busy_rank=pl.col("rides").rank(descending=True),
).head(8)

pickup_ts,rides,prev_day,day_change,running_total,busy_rank
"datetime[μs, UTC]",i64,i64,i64,i64,f64
2026-06-01 00:00:00 UTC,945,null,null,945,13.0
2026-06-02 00:00:00 UTC,1013,945,68,1958,9.0
2026-06-03 00:00:00 UTC,1083,1013,70,3041,3.0
2026-06-04 00:00:00 UTC,1026,1083,-57,4067,8.0
2026-06-05 00:00:00 UTC,1069,1026,43,5136,5.0
2026-06-06 00:00:00 UTC,1007,1069,-62,6143,10.0
2026-06-07 00:00:00 UTC,932,1007,-75,7075,14.0
2026-06-08 00:00:00 UTC,1053,932,121,8128,6.0


## 19. Lazy Execution: scan, plan, collect

Eager Polars runs immediately. Lazy Polars builds a plan first, optimizes it, and runs only when you call `.collect()`.

This is the most important production idea in the notebook.

In [53]:
lazy_report = (
    pl.scan_csv(DATA_PATH, infer_schema=False, null_values=[""])
    .with_columns(
        fare_amount=pl.col("fare_amount").cast(pl.Float64, strict=False),
        tip_amount=pl.col("tip_amount").cast(pl.Float64, strict=False).fill_null(0.0),
        trip_distance=pl.col("trip_distance").cast(pl.Float64, strict=False),
        pickup_ts=pl.col("pickup_ts").str.to_datetime(
            "%Y-%m-%d %H:%M:%S",
            time_zone="UTC",
            strict=False,
        ),
    )
    .drop_nulls(subset=["fare_amount", "pickup_ts", "pickup_borough"])
    .filter(pl.col("fare_amount") > 0)
    .with_columns(
        total_amount=(pl.col("fare_amount") + pl.col("tip_amount")).round(2)
    )
    .group_by("pickup_borough")
    .agg(
        trips=pl.len(),
        revenue=pl.col("total_amount").sum().round(2),
        avg_fare=pl.col("fare_amount").mean().round(2),
    )
    .sort("revenue", descending=True)
)

lazy_report

In [54]:
# The query plan is the teaching moment: Polars can optimize the full pipeline.
print(lazy_report.explain())

SORT BY [descending: [true]] [col("revenue")]
  AGGREGATE[maintain_order: false]
    [len().alias("trips"), col("total_amount").sum().round().alias("revenue"), col("fare_amount").mean().round().alias("avg_fare")] BY [col("pickup_borough")]
    FROM
    simple π 3/3 ["pickup_borough", ... 2 other columns]
       WITH_COLUMNS:
       [[(col("fare_amount")) + (col("tip_amount"))].round().alias("total_amount")] 
        simple π 3/3 ["pickup_borough", ... 2 other columns]
          FILTER [([(col("fare_amount").is_not_null()) & ([(col("fare_amount")) > (0.0)])]) & (col("pickup_ts").is_not_null())]
          FROM
             WITH_COLUMNS:
             [col("fare_amount").cast(Float64), col("tip_amount").cast(Float64).fill_null([0.0]), col("pickup_ts").str.strptime(["raise"])] 
              Csv SCAN [../Labs/Day 1/data/taxi_trip_review.csv]
              PROJECT 4/9 COLUMNS
              SELECTION: col("pickup_borough").is_not_null()
              ESTIMATED ROWS: 17


In [55]:
report = lazy_report.collect()
report

pickup_borough,trips,revenue,avg_fare
str,u32,f64,f64


Instructor bridge: `scan_csv` is closer to how data engineering systems work. You define the plan first, and the engine decides how much data to read, which filters to push down, and how to execute the computation.

## 20. Streaming Engine and Performance Habits

Polars can execute many lazy queries with the streaming engine. For classroom purposes, keep the message simple:

- Use expressions, not Python row loops.
- Prefer lazy scans for file pipelines.
- Project only columns you need.
- Filter early.
- Collect at the boundary.

In [56]:
# Many queries can run with the streaming engine.
# If a specific operation is unsupported, Polars may fall back to in-memory execution.
try:
    lazy_report.collect(engine="streaming")
except TypeError:
    # Compatibility with older Polars versions.
    lazy_report.collect(streaming=True)

In [57]:
# Vectorized expression over a scaled-up frame.
big = pl.concat([clean] * 50_000, how="vertical")
print(f"{big.height:,} rows")

start = perf_counter()
big_total = big.with_columns(
    total_amount=(pl.col("fare_amount") + pl.col("tip_amount")).round(2)
)
elapsed = perf_counter() - start

print(f"expression pipeline: {elapsed:.4f} seconds")
big_total.select("trip_id", "total_amount").head()

0 rows
expression pipeline: 0.0003 seconds


trip_id,total_amount
str,f64


In [58]:
# Anti-pattern demo on a small sample only: Python callbacks do not give Polars much to optimize.
sample = big.head(10_000)

start = perf_counter()
row_result = sample.select(
    pl.struct(["fare_amount", "tip_amount"])
    .map_elements(
        lambda row: row["fare_amount"] + row["tip_amount"],
        return_dtype=pl.Float64,
    )
    .alias("total_amount")
)
elapsed = perf_counter() - start

print(f"Python row callback over {sample.height:,} rows: {elapsed:.4f} seconds")
row_result.head()

Python row callback over 0 rows: 0.0106 seconds


total_amount
f64


Do not over-teach microbenchmarks. The durable lesson is architectural: Polars is fastest when students express work as columnar expressions and let the engine plan execution.

## 21. Under the Hood: Arrow, Parquet, and the Bridge Out

Close the loop: Polars is a DataFrame engine, but data engineering lives in file formats, databases, BI tools, and warehouses. The important bridges are Parquet, Arrow, pandas interop, DuckDB, and SQL-style thinking.

In [59]:
print(f"clean DataFrame estimated size: {clean.estimated_size():,} bytes")
print(f"report DataFrame estimated size: {report.estimated_size():,} bytes")

clean DataFrame estimated size: 0 bytes
report DataFrame estimated size: 0 bytes


In [60]:
out_path = Path("polars_foundations_report.parquet")
report.write_parquet(out_path)

roundtrip = pl.read_parquet(out_path)
roundtrip

pickup_borough,trips,revenue,avg_fare
str,u32,f64,f64


In [61]:
# Optional pandas bridge. This requires pandas/pyarrow in the environment.
try:
    report_pd = report.to_pandas()
    print(type(report_pd))
    display(report_pd)
except Exception as exc:
    print("pandas bridge not available in this environment:")
    print(type(exc).__name__, str(exc).splitlines()[0])

<class 'pandas.DataFrame'>


,pickup_borough,trips,revenue,avg_fare


---
## Wrap-Up: Bridges to Name Out Loud

| Polars foundation | Where it returns |
|---|---|
| `pl.col` expressions | pandas `pd.col`, SQL expressions, Spark column expressions |
| `select`, `filter`, `with_columns` | SQL `SELECT`, `WHERE`, computed columns |
| `group_by().agg()` | SQL `GROUP BY`, BigQuery aggregation |
| `.over(...)` | SQL window functions |
| `join` | SQL joins |
| `pivot` / `unpivot` | BI reporting shapes and tidy data |
| `scan_csv` / lazy plans | production ETL/ELT planning |
| `.collect()` | execution boundary |
| Parquet output | lakehouse/warehouse handoff |

Timing guide if running the full tour: Beginner 25 min, Intermediate 40 min, Advanced 35 min. For a shorter class demo, the load-bearing sections are **2, 6, 8, 11, 13, 14, and 19**.

Suggested follow-up activity: ask students to translate 6 pandas cells from the earlier notebook into Polars using this pattern:

1. Identify columns.
2. Write expressions with `pl.col`.
3. Choose the context: `select`, `with_columns`, `filter`, or `group_by`.
4. Collect only if the pipeline is lazy.